# RTX 3090 dev 서버 분석 — 1.5B / 7B, gpu_only vs hybrid (AFFINITY fix 적용)

**핵심 질문**: i9-12900KF (16 physical / AVX2+VNNI) + RTX 3090 환경에서 affinity fix 적용 후
hybrid inference 가 얼마나 gpu_only 에 근접/멀어지는가? 모델 크기별 차이는?

**실험 환경** (violet 워크스테이션):
- CPU: Intel Core i9-12900KF (8 P-core + 8 E-core = 16 physical / 24 logical, 1 NUMA)
- DRAM: 63 GB DDR5 (~60 GB/s)
- GPU: NVIDIA RTX 3090 24 GB GDDR6X (936 GB/s, FP16 ~71 TFLOPS)
- vLLM hybrid: 1 CPU EngineCore (1 NUMA), **16 physical core strict bind (AFFINITY fix 후)**

**실험 목록** (eval/basic/RTX3090/, 전부 affinity fix 이후 측정):
| # | 디렉토리 | 모델 | 모드 | wall / output tp |
|---|----------|------|------|------|
| G1 | 20260414_054643_G | 1.5B | gpu_only | 8.1s / 7590 tok/s |
| G7 | 20260414_055713_G | 7B | gpu_only | 6.5s / 1908 tok/s |
| H1 | 20260414_143419 | 1.5B | hybrid AFFINITY fix | 23.1s / 2671 tok/s |
| H7 | 20260414_150240 | 7B | hybrid AFFINITY fix | 89.6s / 138 tok/s |

**Affinity fix 핵심:** `_setup_cpu_process_env` 에서 부모로부터 상속된 제한된 affinity ({0}) 를
`sched_getaffinity(1)` (cgroup cpuset.effective) 로 복원. 16 physical core 정상 활용.

**결론 미리 보기:**
- 1.5B hybrid / gpu_only = 2.85× (dev CPU 가 GPU 보다 느려 wall 증가)
- 7B hybrid / gpu_only = 13.7× (weight 4.7× 증가 → CPU side 더 급격히 느려짐, AVX-512/AMX 없어서)
- dev hybrid 는 1.5B 에서만 유의미한 관찰 가능. 7B 는 구조적 이득 불가 (Amdahl)


In [ ]:
# ============================================================
#  Setup
# ============================================================
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE = Path('.')
DIRS = {
    'G1': BASE / '20260414_054643_G_GeForce_RTX_3090_x1_Qwen2.5-1.5B-Instruct',
    'G7': BASE / '20260414_055713_G_GeForce_RTX_3090_x1_Qwen2.5-7B-Instruct',
    'H1': BASE / '20260414_143419_GeForce_RTX_3090_x1_Qwen2.5-1.5B-Instruct',
    'H7': BASE / '20260414_150240_GeForce_RTX_3090_x1_Qwen2.5-7B-Instruct',
}
LABELS = {
    'G1': 'GPU-only 1.5B',
    'G7': 'GPU-only 7B',
    'H1': 'Hybrid 1.5B\n(affinity fix)',
    'H7': 'Hybrid 7B\n(affinity fix)',
}
COLORS = {'G1': '#2196F3', 'G7': '#0D47A1', 'H1': '#4CAF50', 'H7': '#2E7D32'}

def load_json(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

bench, sysinfo = {}, {}
for k, d in DIRS.items():
    jf = 'gpu_only.json' if k.startswith('G') else 'hybrid.json'
    bench[k] = load_json(d / jf)
    sif = d / 'system_info.json'
    if sif.exists():
        sysinfo[k] = load_json(sif)

mon_cpu, mon_gpu = {}, {}
for k, d in DIRS.items():
    mode = 'gpu_only' if k.startswith('G') else 'hybrid'
    for mtype, store in [('cpu', mon_cpu), ('gpu', mon_gpu)]:
        csv_path = d / f'{mode}_monitor_{mtype}.csv'
        store[k] = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()

sys_ref = sysinfo.get('H1') or sysinfo.get('G1') or {}
keys = list(DIRS.keys())
print(f'Loaded {len(keys)} runs: {keys}')


---
## Section 1 — 시스템 하드웨어 프로파일

H100x8 서버의 CPU와 GPU 스펙을 나란히 보여준다. **목표**: CPU도 매우 강력한 연산 자원이라는 것을 수치로 확인.

In [ ]:
cpu_info = sys_ref.get('cpu', {})
gpu_info = sys_ref.get('gpu', {})
numa_info = sys_ref.get('numa', {})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- CPU spec card ---
ax = axes[0]
ax.axis('off')
cpu_rows = [
    ('Model', 'Intel Core i9-12900KF'),
    ('Sockets', '1 (1-NUMA)'),
    ('Physical Cores', '16 (8 P-core + 8 E-core)'),
    ('Logical CPUs', '24 (P: 8×2 HT + E: 8×1)'),
    ('Base Clock', 'P-core 3.2 GHz / E-core 2.4 GHz'),
    ('ISA', 'AVX2 + AVX-VNNI'),
    ('AMX', '❌ not supported'),
    ('AVX-512', '❌ not supported'),
    ('L2 Cache', '14 MiB (P: 1.25 MiB × 8 + E: 2 MiB × 2)'),
    ('L3 Cache', '30 MiB (shared)'),
    ('DRAM', '63 GB DDR5'),
    ('Memory BW', '~60 GB/s (dual channel)'),
    ('BF16 Peak', '~0.8 TFLOPS (AVX2, emulated)'),
]
table = ax.table(
    cellText=cpu_rows,
    colLabels=['Item', 'Value'],
    cellLoc='left', loc='center',
    colWidths=[0.35, 0.65],
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    if c == 1 and r > 0 and ('❌' in cpu_rows[r-1][1]):
        cell.set_text_props(color='#C62828', fontweight='bold')
ax.set_title('CPU: Intel Core i9-12900KF', fontsize=13, fontweight='bold', pad=10)

# --- GPU spec card ---
ax2 = axes[1]
ax2.axis('off')
gpu_rows = [
    ('Model', 'NVIDIA GeForce RTX 3090'),
    ('Count', '1'),
    ('VRAM', '24 GB GDDR6X'),
    ('Memory BW', '936 GB/s'),
    ('CUDA Cores', '10,496'),
    ('Tensor Cores', '328 (3rd gen)'),
    ('FP16 Peak', '~71 TFLOPS'),
    ('BF16 Support', '⚠ partial (Ampere)'),
    ('Interconnect', 'PCIe Gen4 x16'),
]
table2 = ax2.table(
    cellText=gpu_rows,
    colLabels=['Item', 'Value'],
    cellLoc='left', loc='center',
    colWidths=[0.35, 0.65],
)
table2.auto_set_font_size(False)
table2.set_fontsize(10)
table2.scale(1, 1.5)
for (r, c), cell in table2.get_celld().items():
    if r == 0:
        cell.set_facecolor('#6A1B9A')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F3E5F5')
ax2.set_title('GPU: NVIDIA RTX 3090 (×1)', fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


---
## Section 2 — 실험 성능 비교 (핵심 지표)

4개 run의 핵심 성능 지표를 한눈에 비교.

In [ ]:
# Extract key metrics
metrics = {}
for k, b in bench.items():
    if b:
        metrics[k] = {
            'output_tp':    b.get('output_throughput', 0),
            'duration':     b.get('duration', 0),
            'wall_time':    b.get('wall_time_s', 0),
            'tpot_mean':    b.get('mean_tpot_ms', 0),
            'tpot_median':  b.get('median_tpot_ms', 0),
            'tpot_p99':     b.get('p99_tpot_ms', 0),
            'ttft_mean':    b.get('mean_ttft_ms', 0),
            'ttft_p99':     b.get('p99_ttft_ms', 0),
        }

keys = list(metrics.keys())
gpu_tp = metrics['G1']['output_tp']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

def bar_plot(ax, metric, title, unit='', log=False, ref_line=None):
    vals = [metrics[k][metric] for k in keys]
    bars = ax.bar([LABELS[k] for k in keys], vals,
                  color=[COLORS[k] for k in keys], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        if val < 1:
            fmt = f'{val:.3f}'
        elif val < 100:
            fmt = f'{val:.1f}'
        else:
            fmt = f'{val:,.0f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                fmt + (f'\n{unit}' if unit else ''), ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    if ref_line:
        ax.axhline(ref_line, color='navy', linestyle='--', linewidth=1.2, alpha=0.6, label=f'GPU-only: {ref_line:,.0f}')
        ax.legend(fontsize=8)
    if log:
        ax.set_yscale('log')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylim(0, max(vals) * 1.2 if not log else None)
    ax.tick_params(axis='x', labelsize=8)

bar_plot(axes[0], 'output_tp',   'Output Throughput (tok/s)', log=True, ref_line=gpu_tp)
bar_plot(axes[1], 'duration',    'Bench Duration (s) [lower is better]', 's')
bar_plot(axes[2], 'tpot_median', 'TPOT Median (ms) [lower is better]', 'ms')
bar_plot(axes[3], 'tpot_mean',   'TPOT Mean (ms)', 'ms', log=True)
bar_plot(axes[4], 'tpot_p99',    'TPOT P99 (ms) [tail latency]', 'ms', log=True)
bar_plot(axes[5], 'ttft_p99',    'TTFT P99 (ms)', 'ms', log=True)

for ax in axes:
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('H100x8 Benchmark Comparison — Qwen2.5-7B 500 req (128in/128out)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
summary_rows = []
for k in keys:
    m = metrics[k]
    ratio = m['output_tp'] / gpu_tp
    summary_rows.append({
        'Run': LABELS[k].replace('\n', ' '),
        'Duration (s)': f"{m['duration']:.1f}",
        'Wall (s)': f"{m['wall_time']:.0f}",
        'Output TP (tok/s)': f"{m['output_tp']:,.0f}",
        'TP ratio (vs GPU)': f"{ratio:.3f}×",
        'TPOT med (ms)': f"{m['tpot_median']:.1f}",
        'TPOT p99 (ms)': f"{m['tpot_p99']:.0f}",
        'TTFT p99 (ms)': f"{m['ttft_p99']:.0f}",
    })
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

---
## Section 3 — GPU 활용률: H100x8도 7B에서 sub-saturated

GPU가 포화되지 않으면 CPU를 활용할 여지가 없다. 반대로 GPU가 포화될수록 CPU의 가치가 커진다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

for idx, (k, ax) in enumerate(zip(keys, axes.flatten())):
    df = mon_gpu[k]
    if df.empty:
        ax.text(0.5, 0.5, 'No GPU data', ha='center', va='center', transform=ax.transAxes)
        continue

    t = df['elapsed_s']
    gpu_util_cols = [c for c in df.columns if c.endswith('_util_pct') and c.startswith('gpu') and not c.startswith('gpu_avg')]

    # Average GPU util
    avg_col = 'gpu_avg_util_pct'
    if avg_col in df.columns:
        ax.fill_between(t, 0, df[avg_col], alpha=0.35, color=COLORS[k], label='avg util')
        ax.plot(t, df[avg_col], color=COLORS[k], linewidth=1.2)

    # GPU power
    pow_col = 'gpu_avg_power_w'
    ax2r = ax.twinx()
    if pow_col in df.columns:
        ax2r.plot(t, df[pow_col], color='gray', linewidth=0.8, linestyle='--', alpha=0.6, label='avg power')
        ax2r.set_ylabel('Avg Power (W)', color='gray', fontsize=8)
        ax2r.tick_params(axis='y', labelcolor='gray', labelsize=7)

    ax.set_xlabel('Elapsed (s)', fontsize=9)
    ax.set_ylabel('GPU Util (%)', fontsize=9)
    ax.set_ylim(0, 110)
    ax.axhline(100, color='red', linewidth=0.7, linestyle=':', alpha=0.5)

    mean_util = df[avg_col].mean() if avg_col in df else 0
    ax.set_title(f'{LABELS[k].replace(chr(10), " ")} | GPU avg util: {mean_util:.1f}%',
                 fontweight='bold', fontsize=10)

    # Highlight bench window
    bench_dur = bench[k].get('duration', 0)
    t_max = t.max()
    if bench_dur < t_max:
        # Estimate bench start offset
        t_bench_start = t_max - bench_dur - (bench[k].get('wall_time_s', t_max) - bench_dur)

plt.suptitle('GPU Utilization Time Series — 8x H100 Average', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# GPU util summary
print('\n[GPU Utilization Stats]')
for k in keys:
    df = mon_gpu[k]
    if df.empty: continue
    col = 'gpu_avg_util_pct'
    if col in df.columns:
        active = df[col][df[col] > 5]  # exclude idle
        print(f'  {k:3s}: mean={df[col].mean():.1f}%  peak={df[col].max():.1f}%  '
              f'active_mean={active.mean():.1f}% (>{len(active)} samples >5%)')

---
## Section 4 — CPU 활용률: GPU-only 중 완전 유휴 vs Hybrid 중 포화

**핵심**: GPU-only 실행 중 112 physical core가 완전히 놀고 있다. NUMA 별 코어 활용 패턴을 시계열로 확인.

In [ ]:
# RTX 3090 dev 머신은 단일 NUMA. P-core primary (0,2,4,...,14) 와 E-core (16-23) 로 구분.
def get_pe_cols(df):
    """Split P-core primaries (HT=0,2,4,6,8,10,12,14) and E-cores (16-23)."""
    core_cols = [c for c in df.columns if c.startswith('core') and c.endswith('_util_pct')]

    def core_num(c):
        return int(c.replace('core', '').replace('_util_pct', ''))

    p_primary = [c for c in core_cols if core_num(c) in {0, 2, 4, 6, 8, 10, 12, 14}]
    p_ht      = [c for c in core_cols if core_num(c) in {1, 3, 5, 7, 9, 11, 13, 15}]
    e_cores   = [c for c in core_cols if 16 <= core_num(c) <= 23]
    return p_primary, p_ht, e_cores

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for idx, (k, ax) in enumerate(zip(keys, axes.flatten())):
    df = mon_cpu[k]
    if df.empty:
        ax.text(0.5, 0.5, 'No CPU data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{k}: {LABELS[k].replace(chr(92)+"n", " ")}', fontsize=10)
        continue

    t = df['elapsed_s'] if 'elapsed_s' in df.columns else df.index
    p_p, p_ht, e_c = get_pe_cols(df)

    if p_p:
        ax.plot(t, df[p_p].mean(axis=1), color='#1565C0', linewidth=1.5, label='P-core primary (8)')
    if e_c:
        ax.plot(t, df[e_c].mean(axis=1), color='#E65100', linewidth=1.5, label='E-core (8)')
    if p_ht:
        ax.plot(t, df[p_ht].mean(axis=1), color='#9E9E9E', linewidth=1, linestyle='--', alpha=0.7, label='P-core HT sibling (8)')

    overall = df.get('cpu_avg_util_pct')
    if overall is not None:
        ax.plot(t, overall, color='black', linewidth=1.5, label='Overall avg')

    ax.set_title(f'{k}: {LABELS[k].replace(chr(92)+"n", " ")}', fontsize=10)
    ax.set_ylabel('CPU Util %')
    ax.set_xlabel('Elapsed (s)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 105)

plt.suptitle('CPU Core Utilization — P-primary vs P-HT vs E-core (RTX3090 dev)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 5 — NUMA 바인딩 검증: 2-NUMA 올바른 동작 증명

boot log에서 CPU Engine 1 (NUMA 0)과 Engine 2 (NUMA 1)가 서로 다른 물리 코어에 strict bind 되는 것을 확인.

In [ ]:
import re

# Parse boot log (H_FIX run 기준)
boot_log_path = DIRS['H_FIX'] / 'hybrid_server_boot.log'
if boot_log_path.exists():
    with open(boot_log_path) as f:
        boot_lines = f.readlines()
else:
    boot_lines = []

key_markers = ['HYBRID-RESOLVE', 'HYBRID-LAUNCH', 'HYBRID-CPU-ENV', 'HYBRID-CPU-PROC',
               'HYBRID-CPU-WORKER', 'HYBRID-ROUTER-INIT']
extracted = {m: [] for m in key_markers}
for line in boot_lines:
    line_clean = re.sub(r'\x1b\[[0-9;]*m', '', line).strip()
    for m in key_markers:
        if m in line_clean:
            extracted[m].append(line_clean)

# CPU core layout visualization — i9-12900KF (8 P-core with HT + 8 E-core)
fig, ax = plt.subplots(figsize=(14, 3.5))

# P-cores (primary 0,2,...,14 with HT sibling 1,3,...,15)
for i in range(8):
    primary = 2 * i
    sibling = 2 * i + 1
    x = i * 1.5
    ax.add_patch(mpatches.FancyBboxPatch((x, 1.2), 1.2, 0.8,
                                          boxstyle='round,pad=0.05',
                                          facecolor='#1565C0', alpha=0.85,
                                          edgecolor='black'))
    ax.text(x + 0.6, 1.6, f'P{i}\nlog {primary}/{sibling}', ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')

# E-cores (16-23)
for i in range(8):
    logical = 16 + i
    x = i * 1.5
    ax.add_patch(mpatches.FancyBboxPatch((x, 0.1), 1.2, 0.8,
                                          boxstyle='round,pad=0.05',
                                          facecolor='#E65100', alpha=0.85,
                                          edgecolor='black'))
    ax.text(x + 0.6, 0.5, f'E{i}\nlog {logical}', ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')

ax.text(-0.5, 1.6, 'P-core\n(HT×2)', ha='right', va='center', fontsize=10, fontweight='bold', color='#1565C0')
ax.text(-0.5, 0.5, 'E-core\n(HT×1)', ha='right', va='center', fontsize=10, fontweight='bold', color='#E65100')
ax.set_xlim(-2.5, 12.5)
ax.set_ylim(-0.3, 2.3)
ax.axis('off')
ax.set_title('i9-12900KF CPU Layout — 16 physical cores (8P + 8E), 1 NUMA node',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

# Print HYBRID-CPU-WORKER local_omp_cpuid
print('\n[HYBRID-CPU-WORKER] binding markers:')
for line in extracted.get('HYBRID-CPU-WORKER', []):
    if 'local_omp_cpuid' in line or 'thread binding' in line:
        print('  ', line[:200])


---
## Section 6 — Wave-batch 재앙 분석: max_seqs=1 vs max_seqs=16

TPOT 분포 비교로 wave=16이 왜 재앙인지 보여줌. `median_tpot`은 비슷해 보여도 `mean/p99`가 극단적으로 악화됨.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# TPOT distribution (box plot style)
ax = axes[0]
tpot_data = {}
tpot_labels = []
for k in keys:
    b = bench[k]
    # Reconstruct quantile data (min~p99 box)
    med = b.get('median_tpot_ms', 0)
    mean = b.get('mean_tpot_ms', 0)
    p99 = b.get('p99_tpot_ms', 0)
    std = b.get('std_tpot_ms', 0)
    # Approx p25/p75 = mean +/- 0.67*std
    p25 = max(0, mean - 0.67 * std)
    p75 = mean + 0.67 * std
    tpot_data[k] = {'med': med, 'mean': mean, 'p99': p99, 'p25': p25, 'p75': p75, 'std': std}

x_pos = np.arange(len(keys))
w = 0.5
for i, k in enumerate(keys):
    d = tpot_data[k]
    # box: p25-p75
    ax.bar(i, d['p75'] - d['p25'], bottom=d['p25'], width=w,
           color=COLORS[k], alpha=0.7, edgecolor='black', linewidth=0.8)
    # median line
    ax.hlines(d['med'], i - w/2, i + w/2, colors='white', linewidth=2.5)
    ax.hlines(d['med'], i - w/2, i + w/2, colors='black', linewidth=1.2)
    # p99 whisker
    ax.vlines(i, d['p75'], d['p99'], colors=COLORS[k], linewidth=1.5, linestyle='--')
    ax.scatter([i], [d['p99']], color=COLORS[k], s=60, zorder=5, marker='D')
    # Annotations
    ax.text(i, d['p99'] * 1.05, f'P99\n{d["p99"]:.0f}ms', ha='center', fontsize=8, color=COLORS[k], fontweight='bold')
    ax.text(i, d['med'] - d['p25'] * 0.3, f'Med\n{d["med"]:.0f}ms', ha='center', fontsize=7.5, color='white', fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels([LABELS[k] for k in keys], fontsize=8)
ax.set_ylabel('TPOT (ms)')
ax.set_yscale('log')
ax.set_title('TPOT Distribution (box: P25-P75, whisker: P99)', fontweight='bold')
ax.axhline(23, color='navy', linestyle=':', linewidth=1, alpha=0.7, label='GPU-only median 23ms')
ax.legend(fontsize=8)

# Duration comparison + cause analysis
ax2 = axes[1]
run_labels = [LABELS[k].replace('\n', ' ') for k in keys]
durations = [bench[k].get('duration', 0) for k in keys]
bars = ax2.barh(run_labels, durations, color=[COLORS[k] for k in keys],
                edgecolor='white', height=0.5)
for bar, val in zip(bars, durations):
    ax2.text(val + 10, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}s ({val/durations[0]:.0f}×)', va='center', fontsize=9, fontweight='bold')

ax2.set_xlabel('Bench Duration (s)')
ax2.set_title('Bench Duration Comparison (lower is better)', fontweight='bold')
ax2.axvline(durations[0], color='navy', linestyle='--', linewidth=1)

# Cause annotations
annotations = {
    'GPU-only (baseline)': '',
    'Hybrid max_seqs=1 32t': '← CPU tail: 1 req × ~400s decode time (7B on Xeon 8480+ ≈ 0.32 tok/s)',
    'Hybrid wave=16 32t':    '← IPEX FD batch>1 paged-access penalty + chunked_prefill=False serialization → ×5.3',
    'Hybrid max_seqs=1 56t': '← CPU tail: thread=56 (-3.6%), BW-bound: more cores have negligible effect',
}
for i, (label, ann) in enumerate(annotations.items()):
    if ann:
        ax2.text(durations[i] + 10, i - 0.15, ann, va='top', ha='left', fontsize=8, color='#444444')

ax2.set_xlim(0, max(durations) * 1.5)

plt.suptitle('max_seqs=1 vs wave=16 — Why max_seqs=1 Is Always the Right Answer', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# wave=16 back-calculation
print('\n[wave=16 TPOT Back-Calculation]')
b16 = bench.get('H_FIX', {})
n_req = 500
n_cpu_engines = 2
max_seqs = 16
n_cpu = n_cpu_engines * max_seqs  # 32
tpot_gpu_med = bench.get('G1', {}).get('median_tpot_ms', 22.7)
tpot_mean = b16.get('mean_tpot_ms', 1047)
n_gpu = n_req - n_cpu
# gpu avg × n_gpu + cpu_heavy × n_cpu = mean × n_req
# cpu = (mean * n_req - gpu_med * n_gpu) / n_cpu
cpu_implied_tpot = (tpot_mean * n_req - tpot_gpu_med * n_gpu) / n_cpu
print(f'  N_cpu_engines={n_cpu_engines}, max_seqs={max_seqs} -> total CPU slots = {n_cpu}')
print(f'  TPOT mean: {tpot_mean:.0f} ms (p99: {b16.get("p99_tpot_ms", 15966):.0f} ms)')
print(f'  Back-calc: CPU implied TPOT ~ {cpu_implied_tpot:.0f} ms')
print(f'  -> batch>1 KV paged-access penalty: ~{cpu_implied_tpot/tpot_gpu_med:.0f}x slower than single-seq')

---
## Section 7 — CPU 자원 활용 상태 히트맵

Hybrid 실행 중 어느 코어가 실제로 일하는지 시각화. NUMA 0/1 엔진이 각자의 영역에서 일하는 패턴 확인.

In [ ]:
# 대상 벤치마크 선택 (가장 부하가 높은 M_seqs=1 56t 환경 또는 wave=16)
target_k = 'H_FIX' if 'H_FIX' in mon_cpu and not mon_cpu['H_FIX'].empty else 'H_OMP'
df_cpu = mon_cpu[target_k].copy()

if not df_cpu.empty:
    # 0~111까지의 물리 코어 util 추출
    core_cols = [f'core{i}_util_pct' for i in range(24)]
    hm_data = df_cpu[core_cols].T.values
    
    fig, ax = plt.subplots(figsize=(16, 7))
    cax = ax.imshow(hm_data, aspect='auto', cmap='magma', origin='lower', vmin=0, vmax=100, interpolation='none')
    
    ax.set_ylabel('CPU Logical Core ID (0-23)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Elapsed Time (seconds)', fontsize=11, fontweight='bold')
    ax.set_title(f'CPU Core Utilization Heatmap — vLLM Hybrid ({LABELS.get(target_k, target_k)})', fontsize=14, fontweight='bold', pad=15)
    
    # NUMA 0 / NUMA 1 구분선
    ax.axhline(15.5, color='white', linewidth=2.5, linestyle='--')
    
    # y축 라벨 커스텀 (NUMA 0, NUMA 1 명시)
    ax.set_yticks([7.5, 19.5])
    ax.set_yticklabels(['P-cores\n(0-15)', 'E-cores\n(16-23)'], fontsize=10, fontweight='bold')
    
    cbar = fig.colorbar(cax, ax=ax, pad=0.01)
    cbar.set_label('Core Utilization (%)', fontsize=10, fontweight='bold')
    
    # 설명 텍스트 추가
    ax.text(df_cpu.shape[0]*0.01, 16.5, 'E-cores', color='white', fontsize=11, fontweight='bold')
    ax.text(df_cpu.shape[0]*0.01, 1.5, 'P-cores', color='white', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
else:
    print("CPU monitor data not found!")


In [ ]:
def plot_cpu_heatmap(df, title, ax, n_cores=24):
    """CPU per-core utilization heatmap (core x time)"""
    if df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return

    core_cols = sorted(
        [c for c in df.columns if c.startswith('core') and c.endswith('_util_pct')],
        key=lambda c: int(c.replace('core', '').replace('_util_pct', ''))
    )
    # Limit to 112 cores and 200 time points
    core_cols = core_cols[:n_cores]
    df_hm = df[core_cols].T  # shape: (cores, time)
    if df_hm.shape[1] > 200:
        step = df_hm.shape[1] // 200
        df_hm = df_hm.iloc[:, ::step]

    im = ax.imshow(df_hm.values, aspect='auto', vmin=0, vmax=100,
                   cmap='YlOrRd', interpolation='nearest')

    # NUMA boundary line
    ax.axhline(15.5, color='white', linewidth=2.5, linestyle='--')
    ax.text(df_hm.shape[1] * 0.02, 7, 'P-cores\n(0-15)', fontsize=9, color='white', va='center', fontweight='bold')
    ax.text(df_hm.shape[1] * 0.02, 19, 'E-cores\n(16-23)', fontsize=9, color='white', va='center', fontweight='bold')

    ax.set_xlabel('Time ->', fontsize=9)
    ax.set_ylabel('CPU Core Index', fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_yticks([0, 7, 15, 19, 23])
    ax.set_yticklabels(['core 0', 'core 7', 'core 15', 'core 19', 'core 23'], fontsize=7)
    ax.set_xticks([])
    return im

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ims = []
for k, ax, label in zip(['G1', 'H_OMP', 'H_FIX'], axes, ['GPU-only (1.5B)', 'Hybrid OMP-only', 'Hybrid AFFINITY fix']): 
    im = plot_cpu_heatmap(mon_cpu[k], label, ax)
    if im is not None:
        ims.append(im)

if ims:
    cbar = fig.colorbar(ims[-1], ax=axes[-1], orientation='vertical', fraction=0.03, pad=0.03)
    cbar.set_label('CPU Utilization (%)', fontsize=9)

plt.suptitle('CPU Per-Core Utilization Heatmap (24 logical cores x time)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 8 — CPU 연산 잠재력과 현재 병목 분석

Xeon 8480+가 이론적으로 낼 수 있는 성능과, 현재 7B BF16 decode 워크로드에서 실제로 병목이 무엇인지 분석.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Roofline (decode GEMV OI≈1) ---
ax = axes[0]
cpu_peak_bf16 = 0.8e12     # AVX2 BF16 emulated
cpu_bw        = 60e9        # DDR5 dual channel
gpu_peak_fp16 = 71e12       # RTX 3090
gpu_bw        = 936e9

oi_range = np.logspace(-2, 3, 500)
cpu_roof = np.minimum(cpu_peak_bf16, cpu_bw * oi_range)
ax.loglog(oi_range, cpu_roof / 1e12, '-', color='#1565C0', linewidth=2.5,
          label='CPU i9-12900KF (AVX2, 60 GB/s)')
gpu_roof = np.minimum(gpu_peak_fp16, gpu_bw * oi_range)
ax.loglog(oi_range, gpu_roof / 1e12, '-', color='#6A1B9A', linewidth=2.5,
          label='GPU RTX 3090 FP16')

oi_decode = 1.0
cpu_perf = min(cpu_peak_bf16, cpu_bw * oi_decode) / 1e12
gpu_perf = min(gpu_peak_fp16, gpu_bw * oi_decode) / 1e12
ax.scatter([oi_decode], [cpu_perf], s=120, color='#1565C0', zorder=5)
ax.scatter([oi_decode], [gpu_perf], s=120, color='#6A1B9A', zorder=5)
ax.annotate(f'CPU decode\n{cpu_perf:.2f} TF (BW-bound)',
            xy=(oi_decode, cpu_perf), xytext=(0.15, cpu_perf * 3),
            arrowprops=dict(arrowstyle='->', color='#1565C0'), fontsize=9, color='#1565C0')
ax.annotate(f'GPU decode\n{gpu_perf:.1f} TF',
            xy=(oi_decode, gpu_perf), xytext=(5, gpu_perf * 0.5),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A'), fontsize=9, color='#6A1B9A')
ax.set_xlabel('Operational Intensity (FLOPs / Byte)')
ax.set_ylabel('Performance (TFLOPS)')
ax.set_title('Roofline — i9-12900KF vs RTX 3090', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# --- gpu_only vs hybrid (AFFINITY fix) 비교 표 (1.5B + 7B) ---
ax2 = axes[1]
ax2.axis('off')

g1 = bench.get('G1', {}); h1 = bench.get('H1', {})
g7 = bench.get('G7', {}); h7 = bench.get('H7', {})

def ratio(a, b):
    try:
        return f'{a/b:.2f}×' if b > 0 else 'N/A'
    except Exception:
        return 'N/A'

rows = [
    ['Model / Mode', '1.5B gpu_only', '1.5B hybrid fix', '7B gpu_only', '7B hybrid fix'],
    ['Wall (s)', f"{g1.get('duration', 0):.1f}", f"{h1.get('duration', 0):.1f}",
                 f"{g7.get('duration', 0):.1f}", f"{h7.get('duration', 0):.1f}"],
    ['Output TP (tok/s)', f"{g1.get('output_throughput', 0):.0f}", f"{h1.get('output_throughput', 0):.0f}",
                          f"{g7.get('output_throughput', 0):.0f}", f"{h7.get('output_throughput', 0):.0f}"],
    ['TPOT median (ms)', f"{g1.get('median_tpot_ms', 0):.1f}", f"{h1.get('median_tpot_ms', 0):.1f}",
                         f"{g7.get('median_tpot_ms', 0):.1f}", f"{h7.get('median_tpot_ms', 0):.1f}"],
    ['TPOT p99 (ms)', f"{g1.get('p99_tpot_ms', 0):.0f}", f"{h1.get('p99_tpot_ms', 0):.0f}",
                      f"{g7.get('p99_tpot_ms', 0):.0f}", f"{h7.get('p99_tpot_ms', 0):.0f}"],
    ['TTFT p99 (ms)', f"{g1.get('p99_ttft_ms', 0):.0f}", f"{h1.get('p99_ttft_ms', 0):.0f}",
                      f"{g7.get('p99_ttft_ms', 0):.0f}", f"{h7.get('p99_ttft_ms', 0):.0f}"],
    ['hybrid / gpu_only', '—', ratio(h1.get('duration', 0), g1.get('duration', 1)),
                           '—', ratio(h7.get('duration', 0), g7.get('duration', 1))],
]
t2 = ax2.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
               colWidths=[0.24, 0.19, 0.19, 0.19, 0.19])
t2.auto_set_font_size(False); t2.set_fontsize(9.5); t2.scale(1, 1.7)
for (r, c), cell in t2.get_celld().items():
    if r == 0:
        cell.set_facecolor('#37474F'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F5F5F5')
    if c == 0 and r > 0:
        cell.set_text_props(fontweight='bold')
ax2.set_title('AFFINITY fix 후 — 1.5B / 7B × gpu_only vs hybrid', fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


---
## Section 9 — GPU 포화 조건 탐색: 어떤 상황에서 CPU가 실제로 도움이 되나

현재 7B TP=4에서는 GPU가 항상 더 빠르므로 Property 2 gate가 모든 요청을 GPU로 보낸다. GPU를 포화시키면 CPU의 가치가 생긴다.

In [ ]:
# RTX3090 dev 에서 측정한 GPU util 비교 (1.5B vs 7B)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for k in ['G1', 'G7']:
    df = mon_gpu.get(k, pd.DataFrame())
    if df.empty or 'gpu_avg_util_pct' not in df.columns:
        continue
    t = df.get('elapsed_s', df.index)
    ax.plot(t, df['gpu_avg_util_pct'], label=LABELS[k].replace(chr(92)+'n', ' '),
            color=COLORS[k], linewidth=1.3)
ax.set_xlabel('Elapsed (s)')
ax.set_ylabel('GPU Util (%)')
ax.set_title('GPU Utilization — RTX 3090 GPU-only', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

ax2 = axes[1]
# Summary bar chart of mean GPU util
gpu_mean = {}
for k in keys:
    df = mon_gpu.get(k, pd.DataFrame())
    if df.empty or 'gpu_avg_util_pct' not in df.columns:
        gpu_mean[k] = 0
        continue
    gpu_mean[k] = df['gpu_avg_util_pct'].mean()

x = np.arange(len(keys))
bars = ax2.bar(x, [gpu_mean[k] for k in keys],
               color=[COLORS[k] for k in keys], alpha=0.85, edgecolor='white')
for bar, k in zip(bars, keys):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{gpu_mean[k]:.0f}%', ha='center', fontsize=9, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels([LABELS[k].replace(chr(92)+'n', ' ') for k in keys], fontsize=8)
ax2.set_ylabel('Mean GPU Util (%)')
ax2.set_title('Mean GPU Utilization by Run', fontweight='bold')
ax2.axhline(100, color='red', linestyle='--', linewidth=1, label='Saturated')
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


---
## Section 10 — 종합: 현재 상태와 다음 목표

H100x8 서버에서 CPU를 실제로 활용하려면 무엇이 필요한가.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

g1 = bench.get('G1', {}); h1 = bench.get('H1', {})
g7 = bench.get('G7', {}); h7 = bench.get('H7', {})

def ratio(a, b):
    try:
        return f'{a/b:.2f}×' if b > 0 else 'N/A'
    except Exception:
        return 'N/A'

rows = [
    ['Item', 'Value / Status', 'Meaning'],
    ['CPU Resources', '16 physical cores (8P+8E), AVX2+VNNI, no AMX, 63 GB DDR5',
     '⚠ BF16/AMX 없음 → 큰 GEMM 성능 제한'],
    ['affinity fix: sched_setaffinity(0, cgroup.effective)',
     '16 physical cores 전부 사용',
     '✓ H100 은 이미 full 이었으므로 no-op, dev 에서만 효과'],
    ['1.5B gpu_only', f"wall {g1.get('duration', 0):.1f}s / {g1.get('output_throughput', 0):.0f} tok/s", '기준'],
    ['1.5B hybrid fix', f"wall {h1.get('duration', 0):.1f}s / {h1.get('output_throughput', 0):.0f} tok/s",
     f"gpu_only 대비 {ratio(h1.get('duration', 0), g1.get('duration', 1))}"],
    ['7B gpu_only', f"wall {g7.get('duration', 0):.1f}s / {g7.get('output_throughput', 0):.0f} tok/s", '기준'],
    ['7B hybrid fix', f"wall {h7.get('duration', 0):.1f}s / {h7.get('output_throughput', 0):.0f} tok/s",
     f"gpu_only 대비 {ratio(h7.get('duration', 0), g7.get('duration', 1))}"],
    ['Model 크기 영향', '1.5B 2.85× vs 7B 13.7×',
     'weight 4.7× → CPU side 더 급격히 느려짐 (AMX 없으므로)'],
    ['Amdahl 구조적 한계', 'T_hybrid = max(T_gpu_share, T_cpu_share)',
     'dev 에서는 GPU 포화 workload 확보 어려움 → hybrid 이득 불가'],
    ['다음 단계 (TODO §1)', 'A1 spec decode CPU drafter / A2 KV offload / A3 P-D disagg',
     'request-level partition 의 천장 돌파 필요'],
]

table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='left', loc='upper center',
                 colWidths=[0.24, 0.38, 0.38])
table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1, 1.8)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor('#EEEEEE')
    if r == 0:
        cell.set_facecolor('#37474F'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F5F5F5')

ax.set_title('RTX 3090 dev — Affinity Fix 후 종합 요약', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()
